In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder\    #entry point for spark.
        .appName("SparkPractice")\ # label for your job (check ui).
        .master("local[*]")\      # use all cpu cores locally.
        .getOrCreate()


In [ ]:
spark

In [ ]:
data = [
    (1, "2024-01-01", 101, "laptop", 1200, "US"),
    (2, "2024-01-02", 102, "phone", 800, "IN"),
    (3, "2024-01-03", 101, "mouse", 40, "US"),
    (4, "2024-01-04", 103, "keyboard", 100, "UK"),
    (5, "2024-01-05", 104, "monitor", 300, "US"),
    (6, "2024-01-06", 102, "laptop", 1100, "IN"),
    (7, "2024-01-07", 101, "phone", 900, "US")
]

columns = ["order_id", "order_date", "user_id", "product", "amount", "country"]


df = spark.createDataFrame(data,columns)

In [ ]:
df.show()

In [ ]:
df.printSchema()   # no job trigger

In [ ]:
df.describe().show()

In [ ]:
# show only orders where amount > 500
from pyspark.sql.functions import col
df.filter(col("amount")>500).show()



In [ ]:
df.groupBy('country').sum('amount').show()

In [ ]:
from pyspark.sql.types import IntegerType
new_df = df.withColumn('amount',col('amount').cast(IntegerType()))

In [ ]:
new_df.printSchema()

In [ ]:
from pyspark.sql.functions import col, when
new_df.withColumn(
    'amount_category',
    when(col('amount') > 1000, 'High')
    .when((col('amount') <1000) & (col('amount')>500),'Medium')
    .otherwise('Low')
             ).show()


In [ ]:
#TASK 4
#total revenue per user_id and country combination
from pyspark.sql.functions import sum
df.groupBy('user_id','country')\
    .agg(sum("amount").alias("sum"))\
    .show()



In [ ]:
#TASK 5
# Find users with more than 1 order
from pyspark.sql.functions import count
df1 = df.groupBy('user_id').agg(count('order_id').alias('count_of_orders'))

df1.filter(col('count_of_orders')>1).show()

In [ ]:
#TASK 6
#High value users

df1 = df.groupBy('user_id').agg(sum('amount').alias('total_sum'))

df1.filter(col('total_sum')>1500).show()

## NEW DATASET

In [1]:
from pyspark.sql.functions import *
from pyspark.sql import SparkSession

In [17]:
spark = SparkSession.builder\
        .appName('new_test')\
        .master('local[*]')\
        .getOrCreate()
spark

In [3]:
data = [
    (1, "login", "2024-01-01", "US"),
    (1, "purchase", "2024-01-02", "US"),
    (2, "login", "2024-01-01", "IN"),
    (2, "logout", "2024-01-01", "IN"),
    (2, "purchase", "2024-01-03", "IN"),
    (3, "login", "2024-01-02", "UK"),
    (3, "purchase", "2024-01-05", "UK"),
    (3, "purchase", "2024-01-06", "UK"),
    (4, "login", "2024-01-01", "US"),
]

columns = ["user_id", "event_type", "event_date", "country"]

events_df = spark.createDataFrame(data,columns)

events_df.printSchema()

root
 |-- user_id: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_date: string (nullable = true)
 |-- country: string (nullable = true)



In [ ]:
# TASK 7
# total number of events per user

total_events_df = events_df.groupBy(col('user_id')).agg(count(col('event_type')))
total_events_df.show()

In [ ]:
# TASK 8
# users who performed at least 1 purchase


filtered_df = events_df.filter(col('event_type')=='purchase')
# filtered_df.show()

# OR



In [22]:
# TASK 9
# Create a column:
# is_active_user

# Logic:

# user has more than 2 events → "YES"

# otherwise → "NO"
from pyspark.sql.functions import *
from pyspark.sql import * 
filter_df = events_df.groupBy('user_id').agg(count(col('event_type')).alias('event_count'))

joined_df = events_df.join(filter_df,on='user_id',how='left')
# joined_df.show()

joined_df.withColumn(
                'is_active_user',
                when(col('event_count')>2,'YES')
                .otherwise('No')
                      ).show()


+-------+----------+----------+-------+-----------+--------------+
|user_id|event_type|event_date|country|event_count|is_active_user|
+-------+----------+----------+-------+-----------+--------------+
|      1|     login|2024-01-01|     US|          2|            No|
|      1|  purchase|2024-01-02|     US|          2|            No|
|      2|     login|2024-01-01|     IN|          3|           YES|
|      2|    logout|2024-01-01|     IN|          3|           YES|
|      2|  purchase|2024-01-03|     IN|          3|           YES|
|      3|     login|2024-01-02|     UK|          3|           YES|
|      3|  purchase|2024-01-05|     UK|          3|           YES|
|      3|  purchase|2024-01-06|     UK|          3|           YES|
|      4|     login|2024-01-01|     US|          1|            No|
+-------+----------+----------+-------+-----------+--------------+

